# 05 — Donor & Tokenizer Analysis for SALT (evidence before decisions)

**Why this notebook exists:** the init failure chain so far rests on measurements whose
interpretation is contested. Before changing donor or projection method we test every claim.
The thesis line is **SALT** (semantically-selected local maps); global mapping is only a baseline.

**Research questions & pre-registered predictions** (write outcomes into section G):

| # | Question | Test | Prediction if claim TRUE | Prediction if claim FALSE |
|---|----------|------|--------------------------|---------------------------|
| C | Is the donor-soundness METRIC valid? | NeoBERT emb vs FastText-**EN** (positive control) | NeoBERT ≫ 0.2 | NeoBERT ≈ chance → metric broken, retract donor claims |
| D | Is ViDeBERTa's embedding table semantically flat for VI? | soundness raw + centered + isotropy | stays < 0.05 even centered | centered ≫ raw → it was a cone artifact, donor fine |
| D | Is PhoBERT's table semantic (MLM-trained)? | same protocol | > 0.2 | also ~chance → donor swap pointless |
| E | Does OUR pipeline violate SALT's assumption (anchors selected in FastText space ≠ close in donor space)? | anchor coherence + LOO fasttext- vs donor-selection | donor-selection LOO ≫ fasttext-selection | both low → selection space is not the leak |
| F | Can SALT-LLE be tuned to reconstruct known points? | LOO grid ridge × min_anchors × selection | some cell > 0.7 | all < 0.5 → local math itself unstable with 4,990 anchors |

**Context numbers:** v5 LLE measured kNN-preservation 0.024 and LOO median 0.286 (notebook 04).
ViDeBERTa = DeBERTa-v3/GDES, 138GB CulturaX-vi (2023). PhoBERT-base-v2 = RoBERTa/MLM, 20GB (2020).
More data favors ViDeBERTa's encoder; the question here is only the **embedding table + tokenizer**,
because SALT transfers nothing else.


In [3]:
%%capture
!pip install -U transformers datasets safetensors sentencepiece tokenizers pandas matplotlib tqdm fasttext-wheel xformers


In [4]:
import gc, math, sys
from pathlib import Path
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped:', e)

PROJECT_ROOT = Path('/content/drive/MyDrive/SALT3')
sys.path.insert(0, str(PROJECT_ROOT / 'code')); sys.path.insert(0, '/content')

import importlib
import salt3_diagnostics as dx, salt3_donor_analysis as da, salt3_init_forensics as fx
importlib.reload(dx); importlib.reload(da); importlib.reload(fx)
from salt3_common import extract_embedding_weight
from transformers import AutoModel, AutoModelForMaskedLM, AutoTokenizer

V5_INIT_DIR = PROJECT_ROOT / 'init' / 'videberta_salt_init_v5_globalmap_freqbias'
FT_VI_BIN = str(V5_INIT_DIR / 'cc.vi.300.bin')          # already on Drive from earlier runs
FT_EN_BIN = '/content/cc.en.300.bin'                    # local disk: control only, no need to keep

print('Loading the three embedding tables (CPU)...')
neo_tok  = AutoTokenizer.from_pretrained('chandar-lab/NeoBERT', trust_remote_code=True)
neo_emb  = extract_embedding_weight(AutoModelForMaskedLM.from_pretrained(
    'chandar-lab/NeoBERT', trust_remote_code=True)).float().cpu()
vide_tok = AutoTokenizer.from_pretrained('Fsoft-AIC/videberta-base')
vide_emb = extract_embedding_weight(AutoModel.from_pretrained('Fsoft-AIC/videberta-base')).float().cpu()
pho_tok  = AutoTokenizer.from_pretrained('vinai/phobert-base-v2')
pho_emb  = extract_embedding_weight(AutoModel.from_pretrained('vinai/phobert-base-v2')).float().cpu()
gc.collect()
neo_vocab, vide_vocab, pho_vocab = neo_tok.get_vocab(), vide_tok.get_vocab(), pho_tok.get_vocab()
print(f'NeoBERT {tuple(neo_emb.shape)} | ViDeBERTa {tuple(vide_emb.shape)} | PhoBERT {tuple(pho_emb.shape)}')

# v5 anchor pairs (vi donor token -> neobert token) + their word surfaces
ANCHOR_FILES = ['verified_shared_anchors.csv', 'shared_numbers.csv', 'verified_translation_pairs.csv']
frames = [pd.read_csv(V5_INIT_DIR / f, dtype=str)[['videberta_token', 'neobert_token']]
          for f in ANCHOR_FILES if (V5_INIT_DIR / f).exists()]
m = pd.concat(frames).dropna().drop_duplicates('videberta_token')
ANCHOR_MAP = dict(zip(m.videberta_token, m.neobert_token))
ANCHOR_WORDS = [t.replace('▁', '').replace('_', ' ').lower() for t in ANCHOR_MAP]
print(f'anchor pairs: {len(ANCHOR_MAP)}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading the three embedding tables (CPU)...


rotary.py:   0%|          | 0.00/2.58k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/chandar-lab/NeoBERT:
- rotary.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] A new version of the following files was downloaded from https://huggingface.co/chandar-lab/NeoBERT:
- rotary.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/1.31k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


NeoBERT (30522, 768) | ViDeBERTa (128000, 768) | PhoBERT (64001, 768)


model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

anchor pairs: 4990


## A. Tokenizer fit on Vietnamese text
Fertility (tokens per word — lower = vocab matches the language), unk rate, and how much of
the produced token mass a 30,522-row budget keeps. NeoBERT included as the "wrong-language
tokenizer" reference point. NOTE: CulturaX is **unsegmented**, which is exactly what CPT will
see — PhoBERT was designed for word-segmented input, so this measures PhoBERT *as we would
actually use it*, not at its best.


In [5]:
texts = da.stream_culturax_sample(n_docs=500)
rows = []
for name, tok in [('videberta', vide_tok), ('phobert', pho_tok), ('neobert (EN ref)', neo_tok)]:
    r = da.tokenizer_fertility(tok, texts)
    r.update(da.vocab_mass_topn(tok, texts, n_top=30522 - 5))
    r['tokenizer'] = name
    rows.append(r)
tok_df = pd.DataFrame(rows).set_index('tokenizer')[['fertility', 'unk_rate', 'distinct_ids_used', 'topn_mass']]
print(tok_df.round(4))

print('\n── Segmentation examples ──')
for s in fx.FIXED_VI_SENTENCES[:4]:
    print(f'  «{s[:60]}»')
    print('    videberta:', vide_tok.tokenize(s)[:14])
    print('    phobert  :', pho_tok.tokenize(s)[:14])


README.md:   0%|          | 0.00/32.6k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/90 [00:00<?, ?it/s]

  CulturaX sample: 500 docs
                  fertility  unk_rate  distinct_ids_used  topn_mass
tokenizer                                                          
videberta            1.2278    0.0012              11275        1.0
phobert              1.2331    0.0011              11133        1.0
neobert (EN ref)     1.7883    0.0002               3793        1.0

── Segmentation examples ──
  «Việt Nam là một quốc gia nằm ở khu vực Đông Nam Á.»
    videberta: ['▁Việt', '▁Nam', '▁là', '▁một', '▁quốc', '▁gia', '▁nằm', '▁', 'ở', '▁khu', '▁vực', '▁Đông', '▁Nam', '▁Á']
    phobert  : ['Việt', 'Nam', 'là', 'một', 'quốc', 'gia', 'nằm', 'ở', 'khu', 'vực', 'Đông', 'Nam', 'Á.']
  «Hôm nay thời tiết rất đẹp và bầu trời trong xanh.»
    videberta: ['▁Hôm', '▁nay', '▁thời', '▁tiết', '▁rất', '▁đẹp', '▁và', '▁bầu', '▁trời', '▁trong', '▁xanh', '.']
    phobert  : ['Hôm', 'nay', 'thời', 'tiết', 'rất', 'đẹp', 'và', 'bầu', 'trời', 'trong', 'xa@@', 'nh.']
  «Kinh tế Việt Nam tăng trưởng mạnh trong nhữn

## B. Vocab composition & anchor availability
How much SALT-usable material does each vocab hold (clean Vietnamese full words with FastText
vectors), and how many of the existing 4,990 anchor words exist as a single token in each vocab.


In [6]:
import fasttext
ft_vi = fasttext.load_model(FT_VI_BIN)

print('ViDeBERTa (full 128k):')
comp_vide = da.vocab_composition(vide_vocab, 'videberta', ft=ft_vi)
av_vide = da.anchor_surface_availability(ANCHOR_WORDS, vide_vocab, 'videberta')
print('PhoBERT (full 64k):')
comp_pho = da.vocab_composition(pho_vocab, 'phobert', ft=ft_vi)
av_pho = da.anchor_surface_availability(ANCHOR_WORDS, pho_vocab, 'phobert')


ViDeBERTa (full 128k):
  vocab 128,000 | full-word tokens 38,205 | clean VI words 34,827 | FastText-known 31,612 | digits 1,533 | continuation pieces 89,795
  anchor words present as single token: 4117/4990 (82.5%)
PhoBERT (full 64k):
  vocab  64,001 | full-word tokens 45,098 | clean VI words 39,977 | FastText-known 38,269 | digits 1,038 | continuation pieces 18,903
  anchor words present as single token: 4071/4990 (81.6%)


## C. Metric positive control (English) — decides whether section D means anything
Same soundness metric, but NeoBERT embeddings vs FastText-**English**. NeoBERT is MLM-trained,
so its table is known-semantic. **If this scores ≈ chance, the metric is invalid and every
donor-soundness claim is retracted.** ~7GB download, deleted right after.


In [7]:
import os, gzip, shutil, urllib.request
if not os.path.exists(FT_EN_BIN):
    print('downloading cc.en.300 (~4.2GB gz)...')
    urllib.request.urlretrieve('https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.bin.gz', FT_EN_BIN + '.gz')
    with gzip.open(FT_EN_BIN + '.gz', 'rb') as fi, open(FT_EN_BIN, 'wb') as fo:
        shutil.copyfileobj(fi, fo)
    os.remove(FT_EN_BIN + '.gz')
ft_en = fasttext.load_model(FT_EN_BIN)

print('POSITIVE CONTROL — NeoBERT (MLM, English) vs FastText-en:')
control = dx.donor_space_soundness(neo_emb, neo_vocab, ft_en, n=1000, k=10)
METRIC_VALID = max(control['overlap_raw'], control['overlap_raw_centered']) > 0.2
print(f'\n>>> METRIC {"VALIDATED — donor scores below are meaningful" if METRIC_VALID else "INVALID — do NOT interpret section D"} <<<')
del ft_en; gc.collect()


downloading cc.en.300 (~4.2GB gz)...


POSITIVE CONTROL — NeoBERT (MLM, English) vs FastText-en:
── Donor-space soundness (n=1000, k=10, chance≈0.010) ──
  kNN overlap vs FastText: raw 0.352 | syllable-mean 0.352
  mean-centered          : raw 0.360 | syllable-mean 0.360
  verdict (best variant) : DONOR SOUND — geometry carries VI semantics   (gates: >0.2 usable, <0.05 broken)

>>> METRIC VALIDATED — donor scores below are meaningful <<<


8

## D. Donor soundness, properly controlled
Raw + mean-centered kNN overlap vs FastText-vi, plus isotropy. The centered variant answers the
anisotropy objection: if ViDeBERTa's low raw score was a cone artifact, centering rescues it.


In [8]:
print('── ViDeBERTa isotropy ──')
iso_vide = da.embedding_isotropy(vide_emb)
print('── PhoBERT isotropy ──')
iso_pho = da.embedding_isotropy(pho_emb)

print('\nViDeBERTa vs FastText-vi:')
sound_vide = dx.donor_space_soundness(vide_emb, vide_vocab, ft_vi, n=1000, k=10)
print('\nPhoBERT vs FastText-vi:')
sound_pho = dx.donor_space_soundness(pho_emb, pho_vocab, ft_vi, n=1000, k=10)


── ViDeBERTa isotropy ──
  isotropy: mean pairwise cos raw 0.084 | centered 0.000 (reasonably isotropic)
── PhoBERT isotropy ──
  isotropy: mean pairwise cos raw 0.662 | centered -0.000 (ANISOTROPIC cone — raw-cos kNN suspect)

ViDeBERTa vs FastText-vi:
── Donor-space soundness (n=1000, k=10, chance≈0.010) ──
  kNN overlap vs FastText: raw 0.014 | syllable-mean 0.011
  mean-centered          : raw 0.012 | syllable-mean 0.011
  verdict (best variant) : DONOR BROKEN — ~chance vs FastText; swap donor   (gates: >0.2 usable, <0.05 broken)

PhoBERT vs FastText-vi:
── Donor-space soundness (n=1000, k=10, chance≈0.010) ──
  kNN overlap vs FastText: raw 0.074 | syllable-mean 0.052
  mean-centered          : raw 0.127 | syllable-mean 0.076
  verdict (best variant) : WEAK — some signal; projection must be near-lossless to survive   (gates: >0.2 usable, <0.05 broken)


## E. SALT's core assumption in OUR pipeline
SALT fits a local linear map over anchors chosen by semantic similarity. The paper selects
anchors in the model's own space; our pipeline selects in **FastText** space but fits in
**donor** space. Two tests: (1) anchor coherence — are FastText-selected anchors actually
close in donor space? (2) LOO reconstruction with FastText-selection vs donor-space-selection —
if donor-selection reconstructs known anchors far better, the **bridge** is the deviation
to fix, not SALT.


In [9]:
anchor_tokens = [t for t in ANCHOR_MAP if t in vide_vocab]
print('ViDeBERTa:')
coh_vide = da.anchor_coherence(vide_emb, vide_vocab, ft_vi, anchor_tokens, n=300, k=16)
pho_anchor_tokens = [t.replace('▁', '') for t in ANCHOR_MAP if t.replace('▁', '') in pho_vocab]
print('PhoBERT (remapped surfaces):')
coh_pho = da.anchor_coherence(pho_emb, pho_vocab, ft_vi, pho_anchor_tokens, n=300, k=16)

# LOO: selection-space ablation on ViDeBERTa (the real pipeline math, ground truth)
amap_vide = {t: ANCHOR_MAP[t] for t in anchor_tokens if ANCHOR_MAP[t] in neo_vocab}
tv = {t: vide_vocab[t] for t in amap_vide}
ident = {i: i for i in vide_vocab.values()}
loo_ft = dx.leave_one_out_anchor_error(amap_vide, neo_vocab, tv, ident, neo_emb, vide_emb,
                                       ft_vi, max_anchors=1000, selection='fasttext')
loo_dn = dx.leave_one_out_anchor_error(amap_vide, neo_vocab, tv, ident, neo_emb, vide_emb,
                                       ft_vi, max_anchors=1000, selection='donor')

# Same ablation on PhoBERT-as-donor
amap_pho = {t.replace('▁', ''): ANCHOR_MAP[t] for t in ANCHOR_MAP
            if t.replace('▁', '') in pho_vocab and ANCHOR_MAP[t] in neo_vocab}
tp = {t: pho_vocab[t] for t in amap_pho}
ident_p = {i: i for i in pho_vocab.values()}
loo_pho_ft = dx.leave_one_out_anchor_error(amap_pho, neo_vocab, tp, ident_p, neo_emb, pho_emb,
                                           ft_vi, max_anchors=1000, selection='fasttext')
loo_pho_dn = dx.leave_one_out_anchor_error(amap_pho, neo_vocab, tp, ident_p, neo_emb, pho_emb,
                                           ft_vi, max_anchors=1000, selection='donor')


ViDeBERTa:
  anchor coherence (pool 4990, k=16): FT-vs-donor kNN overlap 0.004
  donor-cos of FT-selected anchors 0.063 vs random 0.062 (gain +0.001 -> selection ~RANDOM in donor space — SALT assumption violated)
PhoBERT (remapped surfaces):
  anchor coherence (pool 4311, k=16): FT-vs-donor kNN overlap 0.110
  donor-cos of FT-selected anchors 0.630 vs random 0.586 (gain +0.045 -> selection ~RANDOM in donor space — SALT assumption violated)
── Leave-one-out anchor reconstruction (selection=fasttext) ──
  anchors tested        : 1000
  cosine  mean/median   : 0.248 / 0.215
  cosine  >0.7 fraction : 0.0%   (>0.5: 6.9%)
  norm ratio mean       : 0.326  (1.0 = perfect scale)
  |coef| max  p50/p95   : 0.12 / 0.25  (>>1 = extrapolation)
  verdict               : PROJECTION IS THE LEAK
── Leave-one-out anchor reconstruction (selection=donor) ──
  anchors tested        : 1000
  cosine  mean/median   : 0.128 / 0.124
  cosine  >0.7 fraction : 0.0%   (>0.5: 0.0%)
  norm ratio mean       : 0.431  (

## F. Can SALT-LLE be tuned? (ridge × min-anchors × selection grid)
LOO median cosine across the knobs the pipeline already has. If some cell clears 0.7, SALT-LLE
is salvageable with the right settings — the SALT-faithful outcome we want.


In [10]:
grid = []
for sel in ('fasttext', 'donor'):
    for ridge in (1e-3, 1e-1, 1.0):
        for min_a in (8, 24):
            df = dx.leave_one_out_anchor_error(amap_vide, neo_vocab, tv, ident, neo_emb, vide_emb,
                                               ft_vi, min_anchors=min_a, ridge=ridge,
                                               max_anchors=600, selection=sel, quiet=True)
            ok = df.dropna(subset=['cos'])
            grid.append(dict(selection=sel, ridge=ridge, min_anchors=min_a,
                             loo_median=float(ok.cos.median()), frac_gt07=float((ok.cos > 0.7).mean())))
grid_df = pd.DataFrame(grid).sort_values('loo_median', ascending=False)
print(grid_df.to_string(index=False))
best = grid_df.iloc[0]
print(f"\nbest: selection={best.selection} ridge={best.ridge} min_anchors={best.min_anchors} "
      f"-> LOO median {best.loo_median:.3f} ({'SALVAGEABLE > 0.7' if best.loo_median > 0.7 else 'still leaky'})")


selection  ridge  min_anchors  loo_median  frac_gt07
 fasttext  1.000           24    0.230291   0.003333
 fasttext  0.100           24    0.228300   0.003333
 fasttext  0.001           24    0.227460   0.003333
 fasttext  1.000            8    0.204978   0.000000
 fasttext  0.100            8    0.198015   0.000000
 fasttext  0.001            8    0.196748   0.000000
    donor  1.000            8    0.131065   0.000000
    donor  1.000           24    0.131065   0.000000
    donor  0.100           24    0.126854   0.000000
    donor  0.100            8    0.125713   0.000000
    donor  0.001           24    0.125515   0.000000
    donor  0.001            8    0.123521   0.000000

best: selection=fasttext ridge=1.0 min_anchors=24 -> LOO median 0.230 (still leaky)


## G. Verdict — outcomes vs pre-registered predictions


In [11]:
def med(df):
    return float(df.dropna(subset=['cos']).cos.median())

print('=' * 70); print('DONOR & TOKENIZER ANALYSIS — VERDICT'); print('=' * 70)
print(f"C  metric control (NeoBERT vs FT-en)  : best {max(control['overlap_raw'], control['overlap_raw_centered']):.3f} "
      f"-> metric {'VALID' if METRIC_VALID else 'INVALID — sections D verdicts void'}")
bv = max(sound_vide['overlap_raw'], sound_vide['overlap_syllable'],
         sound_vide['overlap_raw_centered'], sound_vide['overlap_syllable_centered'])
bp = max(sound_pho['overlap_raw'], sound_pho['overlap_syllable'],
         sound_pho['overlap_raw_centered'], sound_pho['overlap_syllable_centered'])
print(f"D  ViDeBERTa soundness (best variant) : {bv:.3f}  | PhoBERT: {bp:.3f}  (gates >0.2 / <0.05)")
print(f"E  anchor coherence gain  vide {coh_vide['donor_cos_of_ft_selected'] - coh_vide['donor_cos_of_random']:+.3f}"
      f" | pho {coh_pho['donor_cos_of_ft_selected'] - coh_pho['donor_cos_of_random']:+.3f}  (>0.1 = FT selection meaningful in donor space)")
print(f"E  LOO vide: fasttext-sel {med(loo_ft):.3f} vs donor-sel {med(loo_dn):.3f}")
print(f"E  LOO pho : fasttext-sel {med(loo_pho_ft):.3f} vs donor-sel {med(loo_pho_dn):.3f}")
print(f"A  fertility vide {tok_df.loc['videberta','fertility']:.2f} | pho {tok_df.loc['phobert','fertility']:.2f}; "
      f"top-30k mass vide {tok_df.loc['videberta','topn_mass']:.1%} | pho {tok_df.loc['phobert','topn_mass']:.1%}")
print(f"B  anchors-as-single-token: vide {av_vide['rate']:.1%} | pho {av_pho['rate']:.1%}")
print('-' * 70)
print('Decision guidance:')
print(' - metric INVALID            -> rebuild metric first; no donor conclusions.')
print(' - vide soundness rescued by centering -> donor fine; fix = selection space / LLE knobs (sec F best cell).')
print(' - vide flat AND pho > 0.2   -> donor is the bottleneck; PhoBERT arm justified BY EVIDENCE.')
print(' - donor-sel LOO >> fasttext-sel       -> FastText bridge is the deviation; switch selection space in 01.')
print('=' * 70)


DONOR & TOKENIZER ANALYSIS — VERDICT
C  metric control (NeoBERT vs FT-en)  : best 0.360 -> metric VALID
D  ViDeBERTa soundness (best variant) : 0.014  | PhoBERT: 0.127  (gates >0.2 / <0.05)
E  anchor coherence gain  vide +0.001 | pho +0.045  (>0.1 = FT selection meaningful in donor space)
E  LOO vide: fasttext-sel 0.215 vs donor-sel 0.124
E  LOO pho : fasttext-sel 0.370 vs donor-sel 0.268
A  fertility vide 1.23 | pho 1.23; top-30k mass vide 100.0% | pho 100.0%
B  anchors-as-single-token: vide 82.5% | pho 81.6%
----------------------------------------------------------------------
Decision guidance:
 - metric INVALID            -> rebuild metric first; no donor conclusions.
 - vide soundness rescued by centering -> donor fine; fix = selection space / LLE knobs (sec F best cell).
 - vide flat AND pho > 0.2   -> donor is the bottleneck; PhoBERT arm justified BY EVIDENCE.
 - donor-sel LOO >> fasttext-sel       -> FastText bridge is the deviation; switch selection space in 01.
